In [1]:
import import_ipynb
import numpy as np
import sys
from itertools import combinations, combinations_with_replacement

from functions import *

In [2]:
"""Create a grid of momenta"""

momentum_grid = [[px, py, pz] for pz in range(-5, 6) for py in range(-5, 6) for px in range(-5, 6)\
                 if (pz==0) and (px>0) and ((np.sqrt(px**2+py**2+pz**2)<=5))]
momentum_grid = np.array(momentum_grid)

In [155]:
"""Convert the momenta and kinetic energy grids to dictionaries"""

label_to_mom_dict = {i: item for i, item in enumerate(momentum_grid)}
#mom_to_label_dict = {tuple(key): item for item, key in label_to_mom_dict.items()}
label_to_kin_dict = {key: np.sqrt(val@val) for key, val in label_to_mom_dict.items()}

In [5]:
"""Create array of allowed momenta for 2->2 scattering"""

# create list of momentum pairs where p1 not distinguishable from p2 and p3 not distinguishable from p4
# remove any p1=p3
all_mom_combinations=list(combinations_with_replacement(list(combinations_with_replacement([*range(35)], 2)), 2))
all_mom_combinations = [[item[0][0], item[0][1], item[1][0], item[1][1]] for item in all_mom_combinations if item[0][0] != item[1][0]]

# select for pairs that conserve momentum and kinetic energy and g-function is nonzero
ok_mom_combinations=[item for item in all_mom_combinations\
 if (np.sum(np.abs(label_to_mom_dict[item[0]]+label_to_mom_dict[item[1]]-label_to_mom_dict[item[2]]-label_to_mom_dict[item[3]])) < 10**-7)\
 and (np.abs(label_to_kin_dict[item[0]]+label_to_kin_dict[item[1]]-label_to_kin_dict[item[2]]-label_to_kin_dict[item[3]]) < 10**-7)\
 and (np.abs(g_factor(label_to_mom_dict[item[0]], label_to_mom_dict[item[1]], label_to_mom_dict[item[2]], label_to_mom_dict[item[3]])) > 10**-7)]

# generate reverse pairings, e.g., (q1, q2, q3, q4) -> (q1, q2, q3, q4), (q3, q4, q1, q2)
mom_combinations = []
for item in ok_mom_combinations: 
    mom_combinations.append(item)
    mom_combinations.append([*item[2:], *item[:2]])
mom_combinations.sort()

mom_combinations=np.array(mom_combinations)

In [156]:
# mom_combinations_dict={tuple(item[:2]): set(item[2:]) for item in mom_combinations}
# for key, item in mom_combinations_dict.items():
#     mom_combinations_dict[key] = mom_combinations_dict[key].union(set(item))

In [228]:
# initial_state = np.array([0,5,10,25,28,33])
# initial_state = np.array([0,5,10,12,20,25,28,33])
initial_state = np.array([0, 5, 9, 10, 11, 21, 25, 25, 28, 33])
#accessed_modes = set(initial_state)

In [229]:
# for item in combinations(accessed_modes, 2):
#     try:
#         accessed_modes=accessed_modes.union(mom_combinations_dict[item])
#     except KeyError:
#         pass
# print(sorted(accessed_modes), print(len(accessed_modes)))

In [230]:
accessed_states = [list(initial_state)]

In [239]:
for state in accessed_states.copy():
    kin = get_kin(accessed_states[0])
    mom = get_mom(accessed_states[0])
    for item in combinations_with_replacement(state, 2):
        try:
            #print(item, mom_combinations_dict[item])
            p3, p4 = tuple(mom_combinations_dict[item])
            p1, p2 = item
            subs_dict = {item: item for item in state}
            subs_dict[p1] = p3
            subs_dict[p2] = p4
            new_state = [subs_dict[item] for item in state]
            new_state.sort()
            if (new_state not in accessed_states) and\
            (np.round(get_kin(new_state), 7)==np.round(kin, 7)) and (np.round(get_mom(new_state), 7)==np.round(mom, 7)).all():
                accessed_states.append(new_state)
        except KeyError:
            pass
print(len(accessed_states))

16


In [251]:
get_kin(np.array([0, 5, 8, 8, 11, 21, 26, 27, 28, 33]))

np.float64(33.38498294773373)

In [252]:
get_mom(np.array([0, 5, 8, 8, 11, 21, 26, 27, 28, 33]))

array([21.,  0.,  0.])

In [244]:
mom_combinations_dict[(12, 20)]

{np.int64(11), np.int64(21)}

In [246]:
get_kin(accessed_states[3])

np.float64(33.38498294773374)

In [245]:
get_kin(accessed_states[0])

np.float64(33.38498294773374)

In [248]:
get_mom(accessed_states[3])

array([21.,  0.,  0.])

In [249]:
get_mom(accessed_states[0])

array([21.,  0.,  0.])

In [128]:
def get_kin(state):
    running_kin = 0
    for i in state:
        running_kin += label_to_kin_dict[i]
    return running_kin

def get_mom(state):
    running_mom = np.zeros(3)
    for i in state:
        running_mom += np.array(label_to_mom_dict[i])
    return running_mom

In [116]:
accessed_states[70]

[np.int64(0),
 np.int64(12),
 np.int64(13),
 np.int64(20),
 np.int64(25),
 np.int64(30),
 np.int64(33),
 np.int64(33)]

In [117]:
for i in accessed_states[70]:
    print(i)

0
12
13
20
25
30
33
33


In [129]:
get_mom(accessed_states[70])

array([16.,  8.,  0.])

In [91]:
running = np.zeros(3)
for i in accessed_states[70]:
    running += np.array(label_to_mom_dict[i])
    print(running)

[ 1. -4.  0.]
[ 3. -5.  0.]
[ 6. -6.  0.]
[ 7. -5.  0.]
[ 9. -3.  0.]
[12.  0.  0.]
[14.  4.  0.]
[16.  8.  0.]


In [89]:
running

array([16.,  8.,  0.])

In [78]:
running = 0
for i in accessed_states[70]:
    running += label_to_kin_dict[i]
    print(label_to_kin_dict[i])
running

4.123105625617661
2.23606797749979
3.1622776601683795
1.4142135623730951
2.8284271247461903
4.242640687119285
4.47213595499958
4.47213595499958


np.float64(26.95100454752356)

In [79]:
accessed_states[70]

[np.int64(0),
 np.int64(12),
 np.int64(13),
 np.int64(20),
 np.int64(25),
 np.int64(30),
 np.int64(33),
 np.int64(33)]

In [109]:
np.array(list(accessed_states))

array([ 0,  1,  3,  5,  8, 10, 11, 12, 13, 20, 21, 22, 25, 27, 28, 30, 32,
       33])

In [28]:
kec_nt = np.array(kec_nt)
Nkec = len(kec_nt)

#ninit = np.sort(np.array([0,5,10,25,28,33])) # N=6
ninit = np.sort(np.array([0,5,10,12,20,25,28,33])) # N=8
#ninit = np.sort(np.array([0,5,8,10,12,20,25,26,28,33])) # N=10

Nn = len(ninit)

pinit = np.sum(ps[ninit], axis=0)
keinit = np.sum(kes[ninit])

In [29]:
print(f'Reference state is {ninit}')
print(f'Reference state total momentum: {pinit}')
print(f'Reference state kinetic energy: {keinit}')

Reference state is [ 0  5 10 12 20 25 28 33]
Reference state total momentum: [16.  0.  0.]
Reference state kinetic energy: 26.95100454752356


In [30]:
binom = np.array(list(combinations([i for i in range(Nn)],2)))
Nbinom = len(binom)

In [31]:
def check(state):
    ans = True
    for i in range(len(state)-2):
        if np.var(state[i:i+3]) < 1e-7:
            ans = False
    return ans

In [32]:
def apply(state):
    newstate = []
    for i in range(Nbinom):
        k = np.array([state[binom[i,0]], state[binom[i,1]]])
        for j in range(Nkec):
            if np.sum(np.abs(k-kec_nt[j,:2]))==0:
                state_i = state.copy()
                state_i[binom[i,0]] = kec_nt[j,2]
                state_i[binom[i,1]] = kec_nt[j,3]
                state_i = np.sort(state_i)
                if check(state_i):
                    newstate.append(state_i)
    return np.array(newstate)

In [33]:
nnewstate = 10  # > 1 to enter the while loop
p_states = np.array([ninit])
newstate1 = np.array([np.zeros(Nn), ninit])
trial = 0

while nnewstate > 1:
    print(f'---------------- H^{trial+1} ----------------')
    newstate2 = np.zeros((1,Nn))
    #newstate3 = np.zeros((1,Nn))
    for i in range(1,len(newstate1)):
        newi = apply(newstate1[i])
        if len(newi) > 0:
            newstate2 = np.append(newstate2, newi, axis=0)
       
    newstate1 = np.array([np.zeros(Nn)])
    for j in range(1,len(newstate2)):
        dist = np.sum(np.abs(p_states - newstate2[j]), axis=1)
        if np.min(dist) > 1e-7:
            p_states = np.append(p_states, [newstate2[j].astype(int)], axis=0)
            newstate1 = np.append(newstate1, [newstate2[j]], axis=0)
    nnewstate = len(newstate1)
    print(f'Number of new states(mod flavor choice) at this round: {nnewstate-1}')
    print(f'Number of states(mod flavor choice) visited so far: {len(p_states)}')
        
    trial += 1

---------------- H^1 ----------------
Number of new states(mod flavor choice) at this round: 4
Number of states(mod flavor choice) visited so far: 5
---------------- H^2 ----------------
Number of new states(mod flavor choice) at this round: 8
Number of states(mod flavor choice) visited so far: 13
---------------- H^3 ----------------
Number of new states(mod flavor choice) at this round: 10
Number of states(mod flavor choice) visited so far: 23
---------------- H^4 ----------------
Number of new states(mod flavor choice) at this round: 6
Number of states(mod flavor choice) visited so far: 29
---------------- H^5 ----------------
Number of new states(mod flavor choice) at this round: 1
Number of states(mod flavor choice) visited so far: 30
---------------- H^6 ----------------
Number of new states(mod flavor choice) at this round: 0
Number of states(mod flavor choice) visited so far: 30


In [34]:
n_states = 0

for i in range(len(p_states)):
    totp = np.sum(ps[p_states[i]], axis=0)
    totke = np.sum(kes[p_states[i]])
    nstate = 2**Nn/4**(Nn-len(set(p_states[i])))
    strs = ' '.join([str(x) for x in p_states[i]]) + '\n'
    n_states += nstate
    if np.sum(np.abs(totp-pinit)) > 1e-7 or np.abs(totke-keinit) > 1e-7:
        print('Total momentum or kinetic energy is not conserved!')

In [35]:
print(f'Number of momentum modes pair with conserved P and E is {len(p_states)}')
print(f'Number of states with conserved P,E, and arbitrary flavor contents is {n_states:.0f}')
bins_visited = np.sort(np.array(list(set(p_states.flatten()))))
print(f'Activated momentum modes:')
print(bins_visited)

Number of momentum modes pair with conserved P and E is 30
Number of states with conserved P,E, and arbitrary flavor contents is 6528
Activated momentum modes:
[ 0  1  3  5  8 10 11 12 13 20 21 22 25 27 28 30 32 33]


In [104]:
len(list(accessed_states))

18